# Milestone 1: Payment Fraud Detection Baseline & Temporal Split

This notebook demonstrates:
1. **Exploratory Data Analysis (EDA)** on simulated mobile-money / payment transactions (~1.5% fraud rate).
2. **Strict Temporal (Out-Of-Time) Train/Validation/Test Split** to eliminate future data leakage.
3. **Domain Risk Feature Engineering** (account velocity, balance discrepancy error, drain ratio).
4. **Classical ML Baselines vs. Gradient Boosted Trees** (Logistic Regression vs. LightGBM/XGBoost).
5. **Cost-Sensitive Optimization** (finding the dollar-minimizing threshold rather than arbitrary 0.5).

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from payment_fraud_detection.data_loader import temporal_train_val_test_split, compute_dataset_summary
from payment_fraud_detection.features import FraudFeatureEngineer
from payment_fraud_detection.models import FraudModelTrainer
from payment_fraud_detection.metrics import find_optimal_cost_threshold, evaluate_model_comprehensive

## 1. Load Data & Inspect Imbalance

In [ ]:
data_path = "../data/raw/synthetic_payments.parquet"
df = pd.read_parquet(data_path)
print(f"Total transactions: {len(df):,}")
print(f"Fraud transactions: {df['isFraud'].sum():,} ({df['isFraud'].mean()*100:.2f}%)")
df.head()

## 2. Execute Strict Temporal (Out-of-Time) Split
Never use random train_test_split on fraud transactions! We split strictly by `step`.

In [ ]:
train_df, val_df, test_df, meta = temporal_train_val_test_split(df, time_col="step", train_ratio=0.70, val_ratio=0.15)
print(f"Train set (time <= {meta['train_cutoff']:.1f}): {len(train_df):,} rows (Fraud: {meta['train_fraud_rate']*100:.2f}%)")
print(f"Val set   (time <= {meta['val_cutoff']:.1f}): {len(val_df):,} rows (Fraud: {meta['val_fraud_rate']*100:.2f}%)")
print(f"Test OOT  (time >  {meta['val_cutoff']:.1f}): {len(test_df):,} rows (Fraud: {meta['test_fraud_rate']*100:.2f}%)")

## 3. Feature Engineering

In [ ]:
fe = FraudFeatureEngineer()
X_train = fe.fit_transform(train_df)
X_val = fe.transform(val_df)
X_test = fe.transform(test_df)

y_train = train_df["isFraud"].to_numpy()
y_val = val_df["isFraud"].to_numpy()
y_test = test_df["isFraud"].to_numpy()
print(f"Features engineered ({len(fe.get_feature_names())}): {fe.get_feature_names()}")

## 4. Train Models & Evaluate Benchmark

In [ ]:
trainer = FraudModelTrainer(random_state=42)
trainer.train_logistic_regression(X_train, y_train)
trainer.train_lightgbm(X_train, y_train, X_val=X_val, y_val=y_val)
trainer.train_xgboost(X_train, y_train)

benchmark = trainer.evaluate_all(X_test, y_test, cost_fn=200.0, cost_fp=15.0)
benchmark[["model_name", "pr_auc", "roc_auc", "recall_at_90_precision", "optimal_threshold", "optimal_total_cost", "pct_savings"]]